
# BANA 310 — Tutorial: Categorical Data Analysis & Visualization (Colab)
**Generated:** 2025-09-16T06:02:36  

This tutorial walks you through categorical EDA step by step using two datasets:
- `customer_purchases.csv` (transactions)
- `customer_feedback.csv` (Likert satisfaction)

Run each cell in order. If a cell errors about a missing CSV, upload the file in Colab's file browser and try again.


## 0) Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8,4)
pd.set_option("display.max_columns", None)

print("Libraries ready ✔")

### (Optional) Upload helper (Colab only)

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # choose customer_purchases.csv and customer_feedback.csv

## 1) Load data

In [ ]:
purchases = pd.read_csv("customer_purchases.csv")
feedback  = pd.read_csv("customer_feedback.csv")

display(purchases.head())
display(feedback.head())

## 2) Identify categorical vs numeric

In [ ]:
cat_cols = purchases.select_dtypes(include="object").columns.tolist()
num_cols = purchases.select_dtypes(include=np.number).columns.tolist()
print("Categorical:", cat_cols)
print("Numeric:", num_cols)

## 3) Ordinal categories (ordered labels)

In [ ]:
age_order = ["18-25","26-35","36-45","46-60"]
purchases["AgeGroup"] = pd.Categorical(purchases["AgeGroup"], categories=age_order, ordered=True)
purchases["AgeGroup"].head()

## 4) Frequency tables & sorted countplots

In [ ]:
purchases["ProductCategory"].value_counts()

In [ ]:
order = purchases["ProductCategory"].value_counts().index
sns.countplot(data=purchases, x="ProductCategory", order=order)
plt.title("Purchases by Product Category")
plt.xticks(rotation=15); plt.show()

## 5) Compare two categorical variables (hue)

In [ ]:
sns.countplot(data=purchases, x="ProductCategory", hue="Gender", order=order)
plt.title("Product Category by Gender")
plt.xticks(rotation=15); plt.show()

## 6) Crosstabs → stacked & row-normalized bars

In [ ]:
ct = pd.crosstab(purchases["Region"], purchases["ProductCategory"])
ct

In [ ]:
ax = ct.plot(kind="bar", stacked=True, figsize=(9,4), colormap="tab20")
plt.title("Region × Product Category — Stacked Counts")
plt.ylabel("Count")
plt.legend(bbox_to_anchor=(1.02,1), loc="upper left", title="ProductCategory")
plt.tight_layout(); plt.show()

In [ ]:
ct_prop = ct.div(ct.sum(axis=1), axis=0)
ax = ct_prop.plot(kind="bar", stacked=True, figsize=(9,4), colormap="tab20")
plt.title("Region × Product Category — Row Proportions")
plt.ylabel("Proportion")
plt.legend(bbox_to_anchor=(1.02,1), loc="upper left", title="ProductCategory")
plt.tight_layout(); plt.show()

## 7) Heatmap of row-normalized proportions

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
sns.heatmap(ct_prop, annot=True, fmt=".0%", cmap="YlGnBu", cbar=True)
plt.title("Region × ProductCategory — Row Proportions (Heatmap)")
plt.xlabel("ProductCategory"); plt.ylabel("Region")
plt.tight_layout(); plt.show()

## 8) Barplot of group means (+ error bars)

In [ ]:
sns.barplot(data=purchases, x="ProductCategory", y="Sales", estimator="mean", errorbar="sd", order=order)
plt.title("Average Sales by Product Category (mean ± sd)")
plt.xticks(rotation=15); plt.ylabel("Average Sales"); plt.show()

## 9) Distribution across categories — box / violin + swarm

In [ ]:
sns.boxplot(data=purchases, x="ProductCategory", y="Sales", order=order)
plt.title("Sales Distribution by Category (Boxplot)")
plt.xticks(rotation=15); plt.show()

In [ ]:
sns.violinplot(data=purchases, x="Gender", y="Sales", inner="quartile", cut=0)
sns.swarmplot(data=purchases, x="Gender", y="Sales", color="black", alpha=0.5)
plt.title("Sales by Gender (Violin + Swarm)"); plt.show()

## 10) Faceting (col and row)

In [ ]:
avg_sales = purchases.groupby(["Region","ProductCategory"], as_index=False)["Sales"].mean()
g = sns.catplot(data=avg_sales, x="ProductCategory", y="Sales", col="Region", kind="bar", height=3.2, aspect=1.1, order=order)
g.set_titles("Region: {col_name}")
for ax in g.axes.flat: ax.set_xticklabels(ax.get_xticklabels(), rotation=20)
g.fig.suptitle("Average Sales by Category — Faceted by Region", y=1.05); plt.show()

In [ ]:
avg_sales2 = purchases.groupby(["Region","Gender","ProductCategory"], as_index=False)["Sales"].mean()
g = sns.catplot(data=avg_sales2, x="ProductCategory", y="Sales", col="Region", row="Gender", kind="bar", height=3.0, aspect=1.1, order=order)
g.set_titles("Region: {col_name} | Gender: {row_name}")
for ax in g.axes.flat: ax.set_xticklabels(ax.get_xticklabels(), rotation=20)
g.fig.suptitle("Average Sales — Faceted by Region (col) × Gender (row)", y=1.02); plt.show()

## 11) Percentage labels helper + Likert example

In [ ]:
def add_pct_labels(ax, is_stacked_proportion=False, min_visible=0.05, fmt="{:.0%}"):
    if is_stacked_proportion:
        for container in ax.containers:
            vals = container.datavalues
            labels = [fmt.format(v) if v >= min_visible else "" for v in vals]
            ax.bar_label(container, labels=labels, label_type="center", fontsize=9)
    else:
        total = sum([p.get_height() for p in ax.patches])
        for p in ax.patches:
            h = p.get_height()
            if total > 0:
                ax.annotate("{:.0%}".format(h/total),
                            (p.get_x() + p.get_width()/2., h),
                            ha="center", va="bottom", fontsize=9, xytext=(0,3), textcoords="offset points")

In [ ]:
feedback["Satisfaction"] = pd.Categorical(
    feedback["Satisfaction"],
    categories=["Very dissatisfied","Dissatisfied","Neutral","Satisfied","Very satisfied"],
    ordered=True
)
sat = pd.crosstab(feedback["Channel"], feedback["Satisfaction"])
sat_prop = sat.div(sat.sum(axis=1), axis=0)

ax = sat_prop.plot(kind="bar", stacked=True, figsize=(9,4), colormap="YlGnBu")
add_pct_labels(ax, is_stacked_proportion=True, min_visible=0.08)
plt.title("Satisfaction Mix by Channel (Proportions with % labels)")
plt.ylabel("Proportion")
plt.legend(title="Satisfaction", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout(); plt.show()

## 12) Side-by-side bars of row proportions

In [ ]:
ct2 = pd.crosstab(purchases["Region"], purchases["ProductCategory"])
ct2_prop = ct2.div(ct2.sum(axis=1), axis=0).reset_index().melt(id_vars="Region", var_name="ProductCategory", value_name="prop")
ax = sns.barplot(data=ct2_prop, x="Region", y="prop", hue="ProductCategory")
plt.title("Region × ProductCategory — Row Proportions (Side-by-Side)")
plt.ylabel("Proportion")
plt.legend(bbox_to_anchor=(1.02,1), loc="upper left", title="ProductCategory")
plt.tight_layout(); plt.show()

## 13) (Bonus) Observed vs Expected share (no equations)

In [ ]:
obs = pd.crosstab(purchases["Region"], purchases["ProductCategory"])
obs_prop = obs / obs.values.sum()
row_marg = obs.sum(axis=1); col_marg = obs.sum(axis=0)
exp = np.outer(row_marg, col_marg) / obs.values.sum()
exp = pd.DataFrame(exp, index=obs.index, columns=obs.columns)
exp_prop = exp / exp.values.sum()
dev = obs_prop - exp_prop

plt.figure(figsize=(8,5))
sns.heatmap(dev, cmap="RdBu", center=0, annot=True, fmt=".1%")
plt.title("Region × ProductCategory — Deviation from Expected Share")
plt.xlabel("ProductCategory"); plt.ylabel("Region")
plt.tight_layout(); plt.show()

## 14) (Optional) Pie & Donut (for exposure only)

In [ ]:
top5 = purchases["ProductCategory"].value_counts().nlargest(5)
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].pie(top5.values, labels=top5.index, autopct="%1.0f%%", startangle=90)
ax[0].set_title("Product Category Share (Pie)")
wedges, texts, autotexts = ax[1].pie(top5.values, labels=top5.index, autopct="%1.0f%%", startangle=90)
centre_circle = plt.Circle((0,0), 0.60, fc="white")
ax[1].add_artist(centre_circle)
ax[1].set_title("Product Category Share (Donut)")
plt.tight_layout(); plt.show()

## 15) Practice (pick 2)


1) Countplot of **AgeGroup** with `hue="Gender"` (use the ordered AgeGroup).  
2) Crosstab `Region × Gender` → **row-proportion** stacked bars with % labels.  
3) Boxplot of **Quantity** by **Region**; then try a violin plot.  
4) Facet **average Sales by ProductCategory** with `col="Gender"` instead of Region.  
5) Sort channels by **Very satisfied** share (descending) and replot the stacked proportions.


## 16) Summary


- Categorical = labels (nominal) or ordered labels (ordinal).  
- 1D: `value_counts`, `countplot` (sorted improves clarity).  
- 2D: `crosstab` → stacked counts or row-normalized proportions; heatmaps for compact overviews.  
- Numeric vs category: bar of means (+ error bars), box/violin/swarm for spread/shape.  
- Use facets to compare multiple groups. Add % labels for readability when helpful.
